# User Story 5 : Classification supervisée et évaluation des modèles

## Définition de X et de la variable cible y à partir des données clusterisées

In [3]:
import pandas as pd
file = r"B:\Machine Learning\Sprint1\YC2_DiabetesTrackAI\data\Clustered_Data.csv"
content = pd.read_csv(file)

x = content.drop("Cluster",axis=1)
y = content["Cluster"]


## Division des données en ensembles d’entraînement et de test

In [20]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from collections import Counter


x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size=0.2,random_state=41
)

print(Counter(y_train))

over = RandomOverSampler(random_state=42)
x_train_res,y_train_res = over.fit_resample(x_train,y_train)

print(Counter(y_train_res))

Counter({1: 336, 0: 278})
Counter({1: 336, 0: 336})


## Entraînement et prédiction avec plusieurs modèles de classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

rfc_model = RandomForestClassifier()
gbc_model = GradientBoostingClassifier()
scv_model = SVC()
dtc_model = DecisionTreeClassifier()
lr_model = LogisticRegression()

models={
    'RandomForestClassifier':rfc_model,
    'GradientBoostingClassifier':gbc_model,
    'SVC':scv_model,
    'DecisionTreeClassifier':dtc_model,
    'LogisticRegression':lr_model
}

for name,model in models.items():
    model.fit(x_train_res,y_train_res)
    y_pred = model.predict(x_test)
    

## Évaluation des modèles avec des métriques de classification

In [36]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

results = []
for name, model in models.items():

    model.fit(x_train_res, y_train_res)
    y_pred = model.predict(x_test)
    
    # Évaluation
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred,output_dict=True)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": report['weighted avg']['precision'],
        "Recall": report['weighted avg']['recall'],
        "F1-score": report['weighted avg']['f1-score'],
        "Confusion Matrix": cm
    })

results_df = pd.DataFrame(results)
results_df.head()


,Model,Accuracy,Precision,Recall,F1-score,Confusion Matrix
0,RandomForestClassifier,0.902597,0.902798,0.902597,0.902660,"[[62, 7], [8, 77]]"
1,GradientBoostingClassifier,0.909091,0.909615,0.909091,0.909199,"[[63, 6], [8, 77]]"
2,SVC,0.948052,0.949545,0.948052,0.948158,"[[67, 2], [6, 79]]"
3,DecisionTreeClassifier,0.831169,0.831169,0.831169,0.831169,"[[56, 13], [13, 72]]"
4,LogisticRegression,0.993506,0.993599,0.993506,0.993511,"[[69, 0], [1, 84]]"


## Validation croisée et optimisation des hyperparamètres des modèles

In [37]:
from sklearn.model_selection import GridSearchCV

models = {
    'RandomForestClassifier': RandomForestClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'SVC': SVC(),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'LogisticRegression': LogisticRegression()
}

param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf', 'poly'],
        'gamma': ['scale', 'auto', 0.1]
    },
    'DecisionTreeClassifier': {
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'LogisticRegression': {
        'penalty': ['l1', 'l2', 'none'],
        'C': [0.1, 1, 10],
        'solver': ['liblinear', 'lbfgs', 'saga']
    }
}

for name, model in models.items():
    grid_search = GridSearchCV(estimator=model, param_grid=param_grids[name], cv=5, scoring='accuracy')
    grid_search.fit(x_train_res, y_train_res)
    
    print(f"Model : {name}")
    print(f"Best parameters : {grid_search.best_params_}")
    print(f"Best cross-validation score : {grid_search.best_score_:.4f}\n")


Model : RandomForestClassifier
Best parameters : {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validation score : 0.9583

Model : GradientBoostingClassifier
Best parameters : {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Best cross-validation score : 0.9539

Model : SVC
Best parameters : {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best cross-validation score : 0.9926

Model : DecisionTreeClassifier
Best parameters : {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best cross-validation score : 0.8690



c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the 

Model : LogisticRegression
Best parameters : {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best cross-validation score : 0.9985



c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\bouch\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the 